# Sensor Models — Range, Bearing, and Camera Projection (Simulation Notebook)

This notebook accompanies the **Sensor Models** chapter and demonstrates:

- **Range** and **bearing** measurements to a landmark in 2D  
- **Measurement noise** and likelihood evaluation  
- A simple **Gauss–Newton** position estimate from range+bearing  
- A basic **observability** check (Jacobian rank) for different measurement sets  
- A minimal **pinhole camera projection** example (3D → image plane)

Dependencies: `numpy`, `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

## 1) Geometry: range vs bearing in 2D

State (robot position):  
$\mathbf{x} = \begin{bmatrix}x \\ y\end{bmatrix}$

Landmark position:  
$\ell = \begin{bmatrix}\ell_x \\ \ell_y\end{bmatrix}$

Range model:
$z_r = \sqrt{(x-\ell_x)^2+(y-\ell_y)^2} + v_r$

Bearing model:
$z_\theta = \operatorname{atan2}(y-\ell_y,\,x-\ell_x) + v_\theta$

Note: $\mathrm{atan2}(y,x)$ returns the signed angle between the positive $x$-axis and the vector $(x,y)$, correctly resolving the quadrant and avoiding division-by-zero ambiguity.

In [ ]:
def h_range(xy, ell):
    dx, dy = xy[0]-ell[0], xy[1]-ell[1]
    return np.sqrt(dx*dx + dy*dy)

def h_bearing(xy, ell):
    dx, dy = xy[0]-ell[0], xy[1]-ell[1]
    return np.arctan2(dy, dx)

def wrap_to_pi(a):
    return (a + np.pi) % (2*np.pi) - np.pi

In [ ]:
# Example setup
ell = np.array([2.0, 1.0])
x_true = np.array([5.0, 3.0])

sigma_r = 0.15          # meters
sigma_th = np.deg2rad(2.0)  # radians

z_r = h_range(x_true, ell) + np.random.randn()*sigma_r
z_th = wrap_to_pi(h_bearing(x_true, ell) + np.random.randn()*sigma_th)

z_r, z_th

In [ ]:
# Plot geometry
fig, ax = plt.subplots()
ax.scatter([ell[0]], [ell[1]], marker="*", s=200, label="Landmark")
ax.scatter([x_true[0]], [x_true[1]], s=80, label="True robot position")

# Range circle (noise-free)
r0 = h_range(x_true, ell)
t = np.linspace(0, 2*np.pi, 300)
ax.plot(ell[0] + r0*np.cos(t), ell[1] + r0*np.sin(t), label="True range circle")

# Bearing ray (noise-free)
th0 = h_bearing(x_true, ell)
ax.plot([ell[0], ell[0]+6*np.cos(th0)], [ell[1], ell[1]+6*np.sin(th0)], label="True bearing ray")

ax.set_aspect("equal", adjustable="box")
ax.grid(True)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Range (circle) vs Bearing (ray) geometry")
ax.legend(loc='upper center', bbox_to_anchor=(0.427, 0.42))
plt.show()

## 2) Likelihood of measurements

Assume independent Gaussian noises:

$ p(z_r|\mathbf{x}) \propto \exp\left(-\frac{(z_r-h_r(\mathbf{x}))^2}{2\sigma_r^2}\right),\qquad
p(z_\theta|\mathbf{x}) \propto \exp\left(-\frac{\mathrm{wrap}(z_\theta-h_\theta(\mathbf{x}))^2}{2\sigma_\theta^2}\right) $

In [ ]:
def loglike_range(zr, xy, ell, sigma):
    e = zr - h_range(xy, ell)
    return -0.5*(e/sigma)**2

def loglike_bearing(zth, xy, ell, sigma):
    e = wrap_to_pi(zth - h_bearing(xy, ell))
    return -0.5*(e/sigma)**2

def loglike_joint(zr, zth, xy, ell, sig_r, sig_th):
    return loglike_range(zr, xy, ell, sig_r) + loglike_bearing(zth, xy, ell, sig_th)

In [ ]:
# Likelihood surface over a grid
xs = np.linspace(0, 8, 160)
ys = np.linspace(-1, 6, 140)
LL = np.zeros((len(ys), len(xs)))

for i, y in enumerate(ys):
    for j, x in enumerate(xs):
        LL[i, j] = loglike_joint(z_r, z_th, np.array([x, y]), ell, sigma_r, sigma_th)

fig, ax = plt.subplots()
im = ax.imshow(LL, extent=[xs[0], xs[-1], ys[0], ys[-1]], origin="lower", aspect="auto")
ax.scatter([ell[0]], [ell[1]], marker="*", s=200, label="Landmark")
ax.scatter([x_true[0]], [x_true[1]], s=70, label="True")
ax.set_title("Log-likelihood surface for (range, bearing)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.colorbar(im, ax=ax, label="log p(z|x) + const")
plt.show()

## 3) Estimating Position from Range + Bearing (Gauss–Newton)

We solve

$$
\min_{\mathbf{x}} \|\mathbf{r}(\mathbf{x})\|^2,
\qquad
\mathbf{r}(\mathbf{x}) =
\begin{bmatrix}
\dfrac{z_r - h_r(\mathbf{x})}{\sigma_r} \\[6pt]
\dfrac{\mathrm{wrap}\!\left(z_\theta - h_\theta(\mathbf{x})\right)}{\sigma_\theta}
\end{bmatrix}.
$$

**Gauss–Newton update**

$$
\Delta \mathbf{x} = (J^\top J)^{-1} J^\top \mathbf{r}.
$$

### Angle Wrapping Function

In angular measurement models, residuals must account for the periodic nature of angles. The wrapping operator maps any angle difference to its principal value:

$$
\mathrm{wrap}(\alpha) = \operatorname{atan2}(\sin \alpha,\;\cos \alpha),
$$

which maps any $\alpha \in \mathbb{R}$ into the interval

$$
(-\pi,\;\pi].
$$

This computes the **shortest signed angular difference**, preventing artificial jumps such as

$$
179^\circ - (-179^\circ) = 358^\circ \quad \text{(incorrect residual)},
$$

while

$$
\mathrm{wrap}(358^\circ) = -2^\circ \quad \text{(correct residual)}.
$$

### Normalized Angular Residual

In estimation algorithms, the wrapped innovation is typically normalized by the measurement noise standard deviation:

$$
r_\theta =
\frac{\mathrm{wrap}\!\left(z_\theta - h_\theta(\mathbf{x})\right)}
{\sigma_\theta},
$$

which represents the angular error in **units of standard deviation**, making it consistent with weighted least-squares and probabilistic estimation.

In [ ]:
def jacobian_range(xy, ell):
    dx, dy = xy[0]-ell[0], xy[1]-ell[1]
    r = np.sqrt(dx*dx + dy*dy) + 1e-12
    # dh/dx = [dx/r, dy/r]
    return np.array([dx/r, dy/r])

def jacobian_bearing(xy, ell):
    dx, dy = xy[0]-ell[0], xy[1]-ell[1]
    q = dx*dx + dy*dy + 1e-12
    # dh/dx = [-dy/q, dx/q]
    return np.array([-dy/q, dx/q])

def gauss_newton_range_bearing(zr, zth, x0, ell, sig_r, sig_th, n_iter=8):
    x = x0.astype(float).copy()
    hist = [x.copy()]
    for _ in range(n_iter):
        r1 = (zr - h_range(x, ell))/sig_r
        r2 = wrap_to_pi(zth - h_bearing(x, ell))/sig_th
        r = np.array([r1, r2])
        J = np.vstack([
            -jacobian_range(x, ell)/sig_r,
            -jacobian_bearing(x, ell)/sig_th
        ])
        # Solve least squares (robust to near singularities)
        dx, *_ = np.linalg.lstsq(J, r, rcond=None)
        x = x + dx
        hist.append(x.copy())
        if np.linalg.norm(dx) < 1e-8:
            break
    return x, np.array(hist)

x0 = np.array([4.0, 0.0])
x_hat, hist = gauss_newton_range_bearing(z_r, z_th, x0, ell, sigma_r, sigma_th)
x_true, x_hat

In [ ]:
# Visualize iterations on likelihood surface
fig, ax = plt.subplots()
im = ax.imshow(LL, extent=[xs[0], xs[-1], ys[0], ys[-1]], origin="lower", aspect="auto")
ax.plot(hist[:,0], hist[:,1], marker="o", label="Gauss–Newton iterates")
ax.scatter([x_true[0]], [x_true[1]], s=70, label="True")
ax.scatter([x_hat[0]], [x_hat[1]], s=70, label="Estimate")
ax.scatter([ell[0]], [ell[1]], marker="*", s=200, label="Landmark")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Gauss–Newton estimate from one landmark (range+bearing)")
ax.legend()
plt.colorbar(im, ax=ax, label="log-likelihood")
plt.show()

## 4) Observability check (local)

For a measurement model \(\mathbf{z}=h(\mathbf{x})\), local observability of the **state components** (at a point) is related to the rank of the Jacobian \(H = \partial h / \partial \mathbf{x}\).

Here we compare:
- Range-only to one landmark
- Bearing-only to one landmark
- Range+bearing to one landmark
- Range-only to **two** landmarks

In [ ]:
def H_range(xy, ell):
    return jacobian_range(xy, ell).reshape(1,2)

def H_bearing(xy, ell):
    return jacobian_bearing(xy, ell).reshape(1,2)

def rank(A, tol=1e-10):
    return np.linalg.matrix_rank(A, tol=tol)

xy0 = np.array([5.0, 3.0])
ell1 = np.array([2.0, 1.0])
ell2 = np.array([1.0, 5.0])

cases = {
    "range-only (1 landmark)": H_range(xy0, ell1),
    "bearing-only (1 landmark)": H_bearing(xy0, ell1),
    "range+bearing (1 landmark)": np.vstack([H_range(xy0, ell1), H_bearing(xy0, ell1)]),
    "range-only (2 landmarks)": np.vstack([H_range(xy0, ell1), H_range(xy0, ell2)]),
}

for k, H in cases.items():
    print(f"{k:28s}  H shape={H.shape}  rank(H)={rank(H)}")

**Interpretation:**  
- One range measurement gives a **circle** of possible \((x,y)\) ⇒ rank 1.  
- One bearing measurement gives a **ray** of possible \((x,y)\) ⇒ rank 1.  
- Range+bearing to one landmark generally pins down \((x,y)\) locally ⇒ rank 2 (except for degenerate configurations).  
- Two independent range measurements to different landmarks can also give rank 2.

## 5) Minimal camera projection model (pinhole)

World point in camera frame:
$\mathbf{p} = \begin{bmatrix}X\\Y\\Z\end{bmatrix},\quad Z>0$

Normalized image coordinates:
$u_n = X/Z,\qquad v_n = Y/Z$

Pixel coordinates (intrinsics: $f_x,f_y,c_x,c_y$):
$u = f_x u_n + c_x,\qquad v = f_y v_n + c_y$

Note: Intrinsics (or camera intrinsics) refer to the set of parameters that describe the internal geometry and optics of a camera, i.e., how 3D rays entering the lens are mapped to pixel coordinates on the image sensor.

In [ ]:
def project_pinhole(P, fx=600.0, fy=600.0, cx=320.0, cy=240.0):
    X, Y, Z = P[...,0], P[...,1], P[...,2]
    u = fx*(X/Z) + cx
    v = fy*(Y/Z) + cy
    return np.stack([u, v], axis=-1)

# Simulate a cube of 3D points in front of the camera
n = 200
P = np.column_stack([
    np.random.uniform(-1.0, 1.0, n),   # X
    np.random.uniform(-0.7, 0.7, n),   # Y
    np.random.uniform(2.0, 5.0, n)     # Z
])

uv = project_pinhole(P)
sigma_px = 1.5
uv_noisy = uv + np.random.randn(*uv.shape)*sigma_px

fig, ax = plt.subplots()
ax.scatter(uv[:,0], uv[:,1], s=10, label="ideal")
ax.scatter(uv_noisy[:,0], uv_noisy[:,1], s=10, label="noisy", alpha=0.6)
ax.invert_yaxis()  # image coordinates
ax.set_xlabel("u [px]")
ax.set_ylabel("v [px]")
ax.set_title("Pinhole projection (ideal vs noisy)")
ax.grid(True)
ax.legend()
plt.show()

## 6) What to modify for your own examples

- Change `ell` and `x_true` to match your scenario (e.g., a robot moving past multiple landmarks).
- Extend to **range-only** or **bearing-only** filtering by using the relevant likelihood function.
- Add a **robot heading** $\psi$ and define a bearing model relative to body frame, e.g.,
$\,z_\theta = \mathrm{wrap}(\operatorname{atan2}(y-\ell_y,x-\ell_x)-\psi)+v$.
- For camera models, add **extrinsics** (rotation+translation) before projection.